# Import Packages

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import input_file_name
from functools import reduce

# Run Utils Functions

In [0]:
%run ../utils/pyutils

# Load Raw Data

## Generate Datasource Path

In [0]:
ds = get_wasbs_path(container = "raw")
files = [f.path for f in dbutils.fs.ls(f"{ds}/sold/all/") if f.path.endswith(".csv")]

In [0]:
dfs = [
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(path)
    for path in files
]

## Define Rename Column Settings

In [0]:
rename_dict = {
    "soldPrice.raw": "soldPrice",
    "rent.raw": "rent",
    "floor.raw": "floor",
    "soldSqmPrice.raw": "soldSqmPrice",
    "soldPriceAbsoluteDiff.raw": "soldPriceAbsoluteDiff",
    "soldPricePercentageDiff.raw": "soldPricePercentageDiff",
    "listPrice.raw": "listPrice",
    "livingArea.raw": "livingArea",
    "rooms.raw": "rooms",
    "__typename": "typeName"
}

## Load

In [0]:
dfs = [
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(path)
         .withColumn("source_file", input_file_name())
    for path in files
]

In [0]:
df_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    dfs
)

In [0]:
cols = ["booliId", 
        "streetAddress", 
        "constructionYear", 
        "objectType", 
        "descriptiveAreaName", 
        "soldPriceType", 
        "daysActive", 
        "soldDate", 
        "latitude", 
        "longitude", 
        "url", 
        "__typename",
        "`soldPrice.raw`", 
        "`floor.raw`", 
        "`soldSqmPrice.raw`", 
        "`soldPriceAbsoluteDiff.raw`", 
        "`soldPricePercentageDiff.raw`", 
        "`listPrice.raw`", 
        "`livingArea.raw`", 
        "`rooms.raw`",
        "rent",
        "source_file"]
df_all = df_all.select(cols)

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(schema) \
    .load(f"{ds}/neighborhoods.csv") \
    .withColumn("source_file", input_file_name())

## Rename Columns

In [0]:
for old_name, new_name in rename_dict.items():
    try:
        df_all = df_all.withColumnRenamed(old_name, new_name)
        print(f"Renamed column {old_name} to {new_name}")
    except Exception as e:
        print(f"Failed to rename column {old_name} to {new_name}: {e}")


## Create Raw View

In [0]:
df_all.createOrReplaceTempView("sold_raw")

# Clean Data

## Subsetting

In [0]:
%sql
create or replace temp view sold_raw_subset as
with
  cte_change_data_types as 
  (
    select distinct
        CAST(booliId AS INT) AS booliId,
        CAST(constructionYear AS INT) AS constructionYear,
        CAST(daysActive AS INT) AS daysActive,
        CAST(soldDate AS DATE) AS soldDate,
        CAST(latitude AS FLOAT) AS latitude,
        CAST(longitude AS FLOAT) AS longitude,
        CAST(url AS STRING) AS url,
        CAST(typeName AS STRING) AS typeName,
        CAST(rent AS INT) AS rent,
        CAST(floor AS DECIMAL) AS floor,
        CAST(soldSqmPrice AS FLOAT) AS soldSqmPrice,
        CAST(livingArea AS FLOAT) AS livingArea,
        CAST(rooms AS FLOAT) AS rooms,
        CAST(listPrice AS INT) AS listPrice,
        CAST(soldPrice AS INT) AS soldPrice,
        CAST(source_file AS INT) AS sourceFileName
    from 
        sold_raw
  )

select
  *
from
  cte_change_data_types
where
  url NOT LIKE '%annons%'
  and constructionYear IS NOT NULL 
  and soldDate IS NOT NULL
  and latitude IS NOT NULL
  and longitude IS NOT NULL
  and soldPrice IS NOT NULL
  and listPrice IS NOT NULL
  and floor IS NOT NULL;

## Deduplicating Raw Data

In [0]:
%sql
create or replace temp view sold_raw_subset_deduplicated as
with
  cte_finding_duplicates as 
  (
    select
      a.*,
      row_number() over (partition by booliId, soldDate order by soldPrice desc) as rn
    from
      sold_raw_subset a
  )

select
  *
from
  cte_finding_duplicates
where
  rn = 1 --Selecting the record with highest sellprice if duplicate values where found in key

# Create Delta Parquet Files & DB

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
ds = get_wasbs_path(container = "silver")
query = f"""
CREATE TABLE IF NOT EXISTS silver.Fact_SoldObjects
(
  booliId INT,
  constructionYear INT,
  daysActive INT,
  soldDate TIMESTAMP,
  latitude FLOAT,
  longitude FLOAT,
  url STRING,
  typeName STRING,
  rent INT,
  floor FLOAT, 
  soldSqmPrice FLOAT,
  livingArea FLOAT,
  rooms FLOAT,
  listPrice INT,
  soldPrice INT,
  sourceFileName STRING
)
USING DELTA
LOCATION '{ds}/soldObjects'
"""
spark.sql(query)

# Merge Data

In [0]:
%sql

MERGE INTO 
  silver.Fact_SoldObjects AS T
USING 
  sold_raw_subset_deduplicated AS S ON 
    T.booliId = S.booliId
    and T.soldDate = S.soldDate
WHEN MATCHED THEN UPDATE SET 
  *
WHEN NOT MATCHED THEN INSERT 
  *

# Logs

In [0]:
%sql
DESCRIBE HISTORY silver.Fact_SoldObjects

In [0]:
%sql
select * from silver.Fact_SoldObjects order by soldDate desc;

# Clean Up Raw Container From Raw Files

In [0]:
# Get the path to the "raw" container
raw_container_path = get_wasbs_path(container="raw")

# Delete all files and directories in the "raw" container
dbutils.fs.rm(f"{raw_container_path}/sold/", recurse = True)